# Why streaming needs a context manager but a regular call doesn't

The core difference: a regular API call **returns a value and is done**. A streaming call **opens a connection and holds it open**.

This notebook builds that intuition from scratch.

---
## 1. A regular function call — no cleanup needed

In [1]:
def get_answer(question):
    """Does its work entirely inside the function. Returns a value. Done."""
    result = f"the answer to '{question}' is 42"
    return result   # nothing left open, nothing to clean up

answer = get_answer('what is the meaning of life')
print(answer)
# after this line, nothing is 'held open'. we just have a string.

the answer to 'what is the meaning of life' is 42


In [2]:
# Retry around a regular call is just a loop — no context manager needed

import time

def call_with_retry(question, max_retries=3):
    for attempt in range(max_retries + 1):
        try:
            return get_answer(question)   # returns a value — loop exits, we're done
        except Exception as e:
            if attempt < max_retries:
                print(f'attempt {attempt + 1} failed, retrying...')
            else:
                raise

result = call_with_retry('what is 2 + 2')
print(result)

the answer to 'what is 2 + 2' is 42


The retry loop is simple because `get_answer()` leaves nothing behind. Call it, get a value, move on.

---
## 2. A streaming call — something is held open

Streaming works differently. Instead of waiting for the full response and returning it all at once, the server sends tokens one by one over an open connection. Your code reads them as they arrive.

That means there's a **connection** that:
- opens when you start streaming
- stays open while you read tokens
- must be closed when you're done

Let's simulate this.

In [3]:
class FakeStream:
    """
    Simulates what the Anthropic SDK's stream object does.
    In reality this wraps an HTTP connection; here it's just a list of tokens.
    """

    def __init__(self, tokens):
        self._tokens = tokens
        self._closed = False
        print('  [FakeStream] connection opened')

    def __iter__(self):
        for token in self._tokens:
            if self._closed:
                raise RuntimeError('reading from a closed stream')
            yield token

    def close(self):
        self._closed = True
        print('  [FakeStream] connection closed')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


def open_stream(text):
    """Simulates client.messages.stream() — returns a stream object, doesn't close it."""
    return FakeStream(text.split())

In [4]:
# The fragile way — what happens if you forget to close

stream = open_stream('hello world how are you')
for token in stream:
    print(f'  token: {token}')
# stream is never closed — in a real HTTP connection this leaks a socket
print(f'stream closed? {stream._closed}')

  [FakeStream] connection opened
  token: hello
  token: world
  token: how
  token: are
  token: you
stream closed? False


In [5]:
# The fragile way with an exception — even worse

try:
    stream = open_stream('hello world how are you')
    for i, token in enumerate(stream):
        print(f'  token: {token}')
        if i == 1:
            raise ValueError('something went wrong mid-stream')
except ValueError:
    print('caught the error')

print(f'stream closed? {stream._closed}')  # still False — leaked

  [FakeStream] connection opened
  token: hello
  token: world
caught the error
stream closed? False


In [ ]:
# The safe way — with guarantees close() is always called

try:
    with open_stream('hello world how are you') as stream:
        for i, token in enumerate(stream):
            print(f'  token: {token}')
            if i == 1:
                raise ValueError('something went wrong mid-stream')
except ValueError:
    print('caught the error')

print(f'stream closed? {stream._closed}')  # True — cleaned up even after exception

**The `with` block guarantees `close()` is called no matter what.** This is why the SDK's `client.messages.stream()` is a context manager — the SDK authors built `__enter__`/`__exit__` into it precisely so you can't accidentally leak the connection.

---
## 3. Why you can't retry a stream with a plain loop

Now try to add retry to the streaming case the same way we did for regular calls.

In [ ]:
class FlakyStream(FakeStream):
    """A stream that fails on the first connection attempt."""
    _attempts = 0

    def __init__(self, tokens):
        FlakyStream._attempts += 1
        if FlakyStream._attempts < 2:
            print('  [FlakyStream] connection failed (simulated 429)')
            raise IOError('rate limited')
        super().__init__(tokens)


def open_flaky_stream(text):
    return FlakyStream(text.split())

In [ ]:
# Attempt 1: retry loop like call_with_retry — but the stream is a context manager
# You can't just return it from inside the loop, because then the `with` block
# would be OUTSIDE the retry loop — meaning cleanup happens but retries don't.

FlakyStream._attempts = 0

def stream_with_retry_broken(text, max_retries=3):
    for attempt in range(max_retries + 1):
        try:
            return open_flaky_stream(text)   # returns the stream object...
            # ...but we haven't entered the `with` block yet!
            # The caller will do `with stream_with_retry_broken(...) as s:`
            # and if __enter__ fails, we're already outside the retry loop.
        except IOError as e:
            if attempt < max_retries:
                print(f'  attempt {attempt + 1} failed, retrying...')
            else:
                raise

# This actually works for the open() failure since open_flaky_stream raises in __init__
# But let's be clear about WHY the class approach is the right solution:
# the retry needs to wrap __enter__, which is what _RetryingStream does.
stream = stream_with_retry_broken('hello world')
print(f'got stream: {stream}')

The real problem becomes clear when you look at the Anthropic SDK: `client.messages.stream()` is lazy — it **doesn't actually open the HTTP connection until you call `__enter__`** (i.e. until you enter the `with` block). So the failure happens inside `__enter__`, which is already outside your retry loop if you just return the object.

This is why `_RetryingStream` puts the retry logic **inside `__enter__`** itself.

---
## 4. The solution: `_RetryingStream` — retry inside `__enter__`

In [ ]:
class RetryingStream:
    """
    Wraps a stream-opening function with retry logic.
    The retry happens inside __enter__, which is where the real connection opens.
    __exit__ delegates to the underlying stream's __exit__ to ensure cleanup.
    """

    def __init__(self, open_fn, *args, **kwargs):
        self._open_fn = open_fn
        self._args    = args
        self._kwargs  = kwargs
        self._stream  = None

    def __enter__(self):
        max_retries = 3
        for attempt in range(max_retries + 1):
            try:
                # open the stream and enter its context — this is where connection happens
                self._stream = self._open_fn(*self._args, **self._kwargs)
                return self._stream.__enter__()   # returns the stream to the caller's `as`
            except IOError as e:
                if attempt < max_retries:
                    print(f'  attempt {attempt + 1} failed ({e}), retrying...')
                else:
                    raise

    def __exit__(self, *args):
        # delegate to the real stream's __exit__ so the connection is always closed
        if self._stream:
            return self._stream.__exit__(*args)


FlakyStream._attempts = 0

with RetryingStream(open_flaky_stream, 'hello world how are you') as stream:
    for token in stream:
        print(f'  token: {token}')

print(f'stream closed? {stream._closed}')

---
## 5. Side by side

```
REGULAR CALL                          STREAMING CALL
────────────────────────────────      ────────────────────────────────
call()                                with open_stream() as stream:
  │                                     │  __enter__() ← connection opens here
  │ does all work inside                │
  │                                     │  read tokens one by one
  │                                     │
  ▼                                     │  __exit__()  ← connection closes here
returns complete value               

nothing held open after return        connection held open the whole time
no cleanup needed                     cleanup (close) must always happen
retry = loop around the call          retry = loop inside __enter__
```

That's why `call_with_retry` is a plain function with a `for` loop, and `stream_with_retry` is a context manager with the retry inside `__enter__`.